# 第4周 · 周一：数据从哪来——从业务动作到数据表

> 难度：⭐⭐⭐ | 定位：理解数据从业务系统到分析数据库的完整流转过程，建立数据全链路认知
> 前置依赖：基础SQL知识 | 后续用途：数仓建模、数据质量排查、与数仓工程师协作
> 面试地位：大厂数据分析面试常考，约占数据工程类问题的20%

---

## 📘 区块1：核心概念与第一性原理

### 1.1 数据流转的设计哲学

每一个数据分析师都写过这样的SQL：
```sql
SELECT city, SUM(amount) AS gmv
FROM orders
WHERE order_date >= '2026-01-01'
GROUP BY city;
```

但你是否想过——`orders`这个表，到底是怎么来的？

它不是"天生就存在"的。你看到的每一行数据，都经历了至少4-5个系统的接力传递：

```
用户在App上下单 
  → App把事件发送到日志服务器 
    → 日志被采集到数据仓库 
      → 原始日志被清洗和结构化 
        → 汇总成"订单表" 
          → 你写SQL查到的那个表
```

理解这个流转过程，是区分"普通SQL写手"和"有数据工程思维的分析师"的关键分水岭。前者只会写SELECT，后者知道数据从哪来、经过了什么处理、可能在哪个环节出问题。

### 1.2 从第一性原理推导：为什么需要"数据搬运"？

假设你是电商公司的CTO，公司只有一个MySQL数据库。这个数据库同时在干两件事：
1. **支撑业务**：用户每下一单，MySQL就INSERT一条记录
2. **支撑分析**：分析师跑`SELECT city, SUM(amount) FROM orders WHERE order_date >= '2026-01-01' GROUP BY city`

现在问题来了——双十一当天，每秒有10000个用户下单。MySQL正在疯狂INSERT。这时候分析师跑了一个分析过去30天GMV的SQL，需要扫描5000万行数据。

**结果**：分析SQL占用了大量CPU和IO，导致INSERT变慢 → 用户下单卡顿 → 用户流失。

这就是**OLTP和OLAP必须分离**的第一性原理——你不能让同一套系统同时"服务用户"和"服务分析师"，因为它们对资源的需求是冲突的：

| 维度 | OLTP（在线事务处理） | OLAP（在线分析处理） |
|------|---------------------|---------------------|
| 用途 | 支撑业务运行 | 支撑数据分析 |
| 举例 | 用户下单时的MySQL数据库 | 分析师写SQL查的Hive/StarRocks |
| 操作类型 | INSERT/UPDATE/DELETE（单行） | SELECT（千万行） |
| 响应要求 | 毫秒级 | 秒到分钟级 |
| 数据量级 | GB级 | TB到PB级 |
| 并发模式 | 高并发小查询 | 低并发大查询 |
| 数据时效 | 实时 | T+1为主 |
| 一致性要求 | 强一致（ACID） | 最终一致 |

所以数据需要从OLTP"搬运"到OLAP，这个搬运过程叫**ETL（Extract-Transform-Load）** 或**ELT（Extract-Load-Transform）**。

### 1.3 数据分析师为什么必须理解数据流转

你可能会想："这是数仓工程师的事，我只要会写SQL就够了。"

但现实是——如果不理解数据流转，你会在这三个场景下"裸奔"：

**场景A：数据延迟了，运营在催**
- "昨天DAU怎么还没出来？"
- 如果你不理解数据流转，你只能说"我去问问数仓"——然后等别人回复
- 如果你理解数据流转，你能判断："现在是T+1上午10点，ETL通常在凌晨3点跑，今天上游日志采集有延迟，预计中午12点前产出"

**场景B：两个数据源的数字对不上**
- "你给我的DAU是102万，但App后台显示98万"
- 不理解数据流转 → "数据错了"
- 理解数据流转 → "App后台是设备维度的实时在线数，DAU是用户维度的T+1离线计算，口径不同导致的4万差异是正常的"

**场景C：你想推动数据建设**
- "这个指标我每天都用，能不能提前到早上8点出？"
- 不理解数据流转 → "这有什么难的，早点跑不就行了"
- 理解数据流转 → "这个指标依赖A表和B表的JOIN，A表8点产出，B表9点产出，所以最早9点才能出——要不我们把B表的调度优先级调高？"

> **本周最重要的认知**：数据分析师不是"SQL Boy"。你的价值和不可替代性不在于"谁写得快"，而在于"谁理解数据背后的业务含义和流转过程"。

### 1.4 数据流转的完整链路

```
┌─────────────────────────────────────────────────────────┐
│                    业务系统层（OLTP）                      │
│  MySQL / Oracle / PostgreSQL / MongoDB                  │
│  用户下单、支付、退款、登录、浏览...                        │
│  特点：实时读写、保证一致性、数据量GB级                      │
└────────────────────┬────────────────────────────────────┘
                     │
                     ▼
┌─────────────────────────────────────────────────────────┐
│                    数据采集层                             │
│  Canal（MySQL binlog订阅）/ Flume / Filebeat             │
│  把业务库的变更实时同步到大数据平台                          │
│  特点：实时/近实时、低延迟、不丢数据                         │
└────────────────────┬────────────────────────────────────┘
                     │
                     ▼
┌─────────────────────────────────────────────────────────┐
│                    消息队列层                             │
│  Kafka / RocketMQ / Pulsar                              │
│  充当数据"缓冲区"，解耦采集和处理                          │
│  特点：高吞吐、可回溯、削峰填谷                             │
└────────────────────┬────────────────────────────────────┘
                     │
                     ▼
┌─────────────────────────────────────────────────────────┐
│                  ETL/ELT 处理层                          │
│  Spark / Flink / Hive / DataX                           │
│  清洗、转换、聚合、关联                                    │
│  特点：批处理/流处理、计算密集                              │
└────────────────────┬────────────────────────────────────┘
                     │
                     ▼
┌─────────────────────────────────────────────────────────┐
│                    数据存储层（OLAP）                      │
│  Hive / StarRocks / ClickHouse / Doris                  │
│  分层存储：ODS → DWD → DWS → ADS                        │
│  特点：海量存储、列式存储、适合扫描                         │
└────────────────────┬────────────────────────────────────┘
                     │
                     ▼
┌─────────────────────────────────────────────────────────┐
│                    数据应用层                             │
│  BI看板 / 报表 / 数据产品 / 算法模型 / 你的SQL             │
│  特点：面向业务、可视化、决策支持                           │
└─────────────────────────────────────────────────────────┘
```

每一层都有其独特的设计目标和技术约束。理解每一层的职责，你才能在排查问题时快速定位。

### 1.5 ETL vs ELT：两种数据集成范式

**ETL（Extract-Transform-Load）**：
- 流程：从业务库抽取 → 在中间层清洗转换 → 加载到数仓
- 优点：数据到数仓时已经干净，分析师直接用
- 缺点：转换逻辑变更需要重跑，灵活性差
- 适用：传统数仓（Teradata、Oracle DW）

**ELT（Extract-Load-Transform）**：
- 流程：从业务库抽取 → 直接加载到数仓（原始数据） → 在数仓内转换
- 优点：数据先落地再处理，灵活；计算可以反复调整
- 缺点：数仓需要存储原始脏数据，存储成本高
- 适用：现代大数据数仓（Hive、Spark）

现在大多数互联网公司采用ELT模式——先把数据搬过来，再在数仓内部清洗和聚合。这样你可以随时用原始数据验证你的清洗逻辑是否正确。

### 1.6 实时数仓 vs 离线数仓

离线数仓（T+1）是主流，但越来越多的场景需要"实时"——比如大促实时看板、风控实时识别。

| 维度 | 离线数仓 | 实时数仓 |
|------|---------|----------|
| 延迟 | T+1（小时到天） | 秒到分钟 |
| 技术栈 | Hive + Spark | Flink + Kafka + ClickHouse |
| 成本 | 低 | 高（3-10倍） |
| 适用场景 | 日报、周报、离线分析 | 实时看板、风控、推荐 |
| 数据质量 | 高（可以事后修正） | 中（流式处理可能丢数据） |

**关键判断**：不是所有指标都需要实时。"CEO想看实时GMV"听起来很酷，但如果GMV实时变化导致CEO每5分钟就要看一次——这不是数据赋能，这是数据骚扰。

### 1.7 面试高频考点

**面试官常问**：
- "你们公司的数据是怎么从业务库到数仓的？"
- "OLTP和OLAP的区别是什么？"
- "为什么不能直接在业务数据库上跑分析SQL？"
- "ETL和ELT的区别？"
- "T+1数据和实时数据的区别和适用场景？"

**回答要点**：
- 描述完整链路：业务库 → 采集 → Kafka → ETL → 数仓分层 → 应用
- OLTP面向业务运行（毫秒级读写），OLAP面向数据分析（批量扫描）
- 分析SQL会占用大量资源影响业务正常运行
- ETL先转换再加载，ELT先加载再转换，现代数仓多用ELT
- T+1适用于日常分析，实时适用于监控和风控

## 🧩 区块2：知识点详细拆解

### 2.1 数据采集的四种方式

**为什么重要**：不同的采集方式决定了数据的时效性、完整性和可靠性。

**方式1：数据库Binlog同步（最常用）**
- 原理：MySQL/Oracle的每一次INSERT/UPDATE/DELETE都会被记录到binlog（二进制日志）中。Canal等工具实时"监听"binlog的变化，把变更同步到Kafka
- 优点：实时、不丢数据、对业务库几乎无影响
- 缺点：只能捕获"有主键的表"的行级变更
- 适用：订单表、用户表、支付表等核心业务表

**方式2：API拉取**
- 原理：定时调用业务系统的HTTP API，拉取数据
- 优点：灵活、可以控制频率
- 缺点：有延迟、可能漏数据、对业务系统有压力
- 适用：第三方平台数据（如广告投放数据）

**方式3：日志文件采集**
- 原理：App/Web端把用户行为以日志形式写到文件，Flume/Filebeat采集这些文件
- 优点：高吞吐、适合埋点数据
- 缺点：客户端可能丢日志（用户断网）
- 适用：用户行为埋点、点击流数据

**方式4：数据库直连抽取（JDBC）**
- 原理：ETL工具通过JDBC直接连接业务库，执行SELECT查询抽取数据
- 优点：简单直接
- 缺点：对业务库有压力、不支持实时
- 适用：小数据量的维表同步

### 2.2 消息队列（Kafka）的核心作用

**为什么重要**：Kafka是现代数据架构的"中枢神经"，几乎所有数据都要经过它。

Kafka在数据流转中的三个关键作用：

**1. 削峰填谷**
- 双十一期间，每秒有10万条下单事件涌入
- 如果直接写入Hive，Hive承受不住
- Kafka作为"蓄水池"，暂存这些事件，下游按自己的节奏消费

**2. 解耦生产者和消费者**
- 数据生产者（Canal/Flume）只需要把数据写到Kafka
- 数据消费者（Spark/Flink/Hive）只需要从Kafka读数据
- 双方互不感知，任何一个挂了都不影响对方

**3. 多订阅**
- 同一份订单数据，实时看板需要（Flink消费）、离线数仓需要（Spark消费）、推荐算法也需要（另一个Spark消费）
- Kafka可以让多个消费者各自消费同一份数据，互不影响

### 2.3 数据质量在流转中是如何损失的

**为什么重要**：数据质量问题是分析师的日常噩梦，而根因往往在流转过程中。

**常见的"数据损失"场景**：

| 环节 | 可能的问题 | 表现 |
|------|-----------|------|
| 客户端上报 | 用户断网导致日志丢失 | 部分用户行为无记录 |
| 日志采集 | Flume Agent宕机 | 某段时间的数据缺失 |
| Kafka | 消息积压被丢弃 | 峰值期间数据丢失 |
| ETL | 字段类型转换错误 | 字符串被截断、数字变NULL |
| ETL | JOIN逻辑错误 | 行数膨胀或丢失 |
| 数仓存储 | 分区数据被误删 | 历史数据空洞 |

**排查思路**：
1. 先确认数据在哪个环节出问题（逐层对比数据量）
2. 对比业务库直接查询 vs 数仓查询的结果
3. 检查ETL任务的执行日志
4. 确认上游依赖的表是否正常产出

### 2.4 数据时效性：T+0、T+1、小时级

**为什么重要**：不同的分析需求对时效性要求不同，选错了会产生误导。

| 时效级别 | 延迟 | 典型场景 | 技术实现 |
|---------|------|---------|----------|
| T+0（实时） | 秒级 | 大促实时看板、风控 | Flink + Kafka |
| 小时级 | 1-3小时 | 当日累计数据 | Spark Streaming |
| T+1上午 | 次日8-10点 | 日常日报 | Spark Batch + 分区表 |
| T+1下午 | 次日14-16点 | 复杂报表（多表JOIN） | Hive MR |
| T+2 | 后天 | 月度/季度报表 | Hive MR |

**关键认知**：更实时 ≠ 更好。实时数据通常没有经过完整的清洗和修正，可能存在脏数据。T+1的数据虽然"慢"，但经过了充分的校验和修正，质量更高。

### 2.5 数据血缘（Data Lineage）

**为什么重要**：当某个上游表变更时，你需要知道会影响哪些下游报表。

数据血缘追踪的是"数据从哪来，经过了哪些处理，最终去了哪"。

**例子**：
```
ods_order_log（原始订单日志）
  → dwd_order_detail（清洗后的订单明细）
    → dws_category_daily（品类日汇总）
      → ads_ceo_dashboard（CEO看板）
      → ads_operation_report（运营日报）
    → dws_user_daily（用户日汇总）
      → ads_user_growth_report（用户增长看板）
```

如果`dwd_order_detail`的"退款金额"字段口径变了，所有下游的看板和报表都会受影响。没有血缘追踪，你根本不知道"改了这个会影响什么"。

### 2.6 数据分层的好处

**为什么重要**：分层是数仓设计的核心思想，理解分层才能高效使用数仓。

不分层的问题——如果所有数据都是"原始日志直接堆在那"：
1. **数据太脏**：日志里大量乱码、空值、格式不一致，每个人都要自己清洗
2. **计算太慢**：每次分析都要从头清洗+计算
3. **口径太乱**：每个人自己清洗，同一个指标算出N个结果
4. **依赖混乱**：不知道哪个表依赖哪个表，改一个影响一片

分层的本质：**把"清洗"和"计算"的工作在数据入库时就做好，让分析师拿到手的就是"干净且口径统一"的数据。**

### 2.7 常见的数据采集工具对比

| 工具 | 类型 | 适用场景 | 优势 | 劣势 |
|------|------|---------|------|------|
| Canal | Binlog订阅 | MySQL实时同步 | 实时、低延迟 | 仅支持MySQL |
| DataX | JDBC抽取 | 离线批量同步 | 支持多种数据源 | 不支持实时 |
| Flume | 日志采集 | 日志文件采集 | 高吞吐、可靠 | 配置复杂 |
| Flink CDC | CDC采集 | 实时数据库同步 | 支持多种数据库 | 学习成本高 |
| Sqoop | JDBC抽取 | Hadoop生态 | 与Hive集成好 | 已逐渐被替代 |

### 2.8 数据仓库 vs 数据湖 vs 湖仓一体

**为什么重要**：这是现代数据架构的核心概念，面试中经常被问到。

| 概念 | 特点 | 适用场景 |
|------|------|----------|
| 数据仓库 | 结构化数据、Schema-on-Write、分层设计 | 传统BI和报表 |
| 数据湖 | 全类型数据、Schema-on-Read、无分层 | 数据科学和探索 |
| 湖仓一体 | 数据湖的灵活性 + 数仓的管理能力 | 现代数据平台 |

**数据湖的核心理念**：先把所有数据（结构化的、半结构化的、非结构化的）以原始格式存起来，需要的时候再定义结构。这对数据科学家非常友好——他们可以用原始数据做各种实验。但对分析师不太友好——你需要自己"定义结构"。

## 💻 区块3：动手练习

### 步骤0：准备——理解你的数据环境

In [ ]:
# ===== 步骤0：盘点你日常接触的数据表 =====

print("=" * 60)
print("任务：盘点你的'数据资产'")
print("=" * 60)
print()

# 填写你日常工作中最常用的5张表
tables = [
    ("（填表名）", "这张表从哪个系统来？", "属于ODS/DWD/DWS/ADS哪层？", "T+1还是实时？"),
    ("（填表名）", "这张表从哪个系统来？", "属于ODS/DWD/DWS/ADS哪层？", "T+1还是实时？"),
    ("（填表名）", "这张表从哪个系统来？", "属于ODS/DWD/DWS/ADS哪层？", "T+1还是实时？"),
    ("（填表名）", "这张表从哪个系统来？", "属于ODS/DWD/DWS/ADS哪层？", "T+1还是实时？"),
    ("（填表名）", "这张表从哪个系统来？", "属于ODS/DWD/DWS/ADS哪层？", "T+1还是实时？"),
]

for i, (name, source, layer, freshness) in enumerate(tables, 1):
    print(f"表{i}：{name}")
    print(f"  来源系统：{source}")
    print(f"  数仓层级：{layer}")
    print(f"  数据时效：{freshness}")
    print()

print("如果你填不出上面任何一行——建议去问数仓同事。")
print("了解数据的'来龙去脉'是分析师的基本功。")

### 步骤1：基础练习——数据流转路径绘制

In [ ]:
# ===== 步骤1：绘制你日常使用的某张表的数据流转路径 =====

print("=" * 60)
print("以你最常用的一张分析表为例，追溯它的'前世今生'")
print("=" * 60)
print()

# 示例：假设你最常用 dws_user_daily_1d（用户日活汇总表）
print("示例：dws_user_daily_1d 的数据流转路径")
print()
print("第1步 · 业务系统")
print("  → App埋点上报用户启动事件")
print("  → 后端记录用户登录日志到MySQL user_login_log表")
print()
print("第2步 · 数据采集")
print("  → Canal监听MySQL binlog")
print("  → 把user_login_log的变更实时发送到Kafka topic: user_login")
print()
print("第3步 · ETL处理")
print("  → Spark任务每10分钟消费Kafka，写入Hive ODS层")
print("  → 每天凌晨3点，ETL任务把ODS数据清洗写入DWD层")
print("  → 凌晨4点，汇总任务把DWD数据聚合写入DWS层")
print()
print("第4步 · 数据应用")
print("  → 早上8点，你的SQL：SELECT * FROM dws_user_daily_1d WHERE dt = '昨天'")
print("  → 得到结果，写日报")
print()

# 现在轮到你了
print("=" * 60)
print("现在请你画出你最常用的一张表的流转路径：")
print()
print("表名：（填）")
print()
print("第1步 · 业务系统 → （填：哪个系统的哪个表/哪个埋点）")
print("第2步 · 数据采集 → （填：Canal？Flume？API？）")
print("第3步 · ETL处理 → （填：经过了哪些转换和汇总）")
print("第4步 · 数据应用 → （填：你用它做什么分析/报表）")
print()
print("提示：如果你画不出来——这就是你今天最重要的TODO。")

### 步骤2：进阶练习——数据延迟排查与数据质量检查

In [ ]:
# ===== 步骤2.1：数据延迟排查清单 =====

print("=" * 60)
print("场景模拟：今天早上10点，运营来问"昨天DAU还没出来"")
print("=" * 60)
print()

# 延迟排查步骤
print("排查清单（从上到下逐层检查）：")
print()

checks = [
    ("① 检查看板/报表", "看板数据源是否配置正确？刷新时间是否过了？"),
    ("② 检查ADS/应用层", "ads层的表有没有产出？查看该表的最新分区"),
    ("③ 检查DWS/汇总层", "dws层的上游表有没有产出？查看ETL任务执行时间"),
    ("④ 检查DWD/明细层", "dwd层的清洗任务有没有跑完？是否有报错？"),
    ("⑤ 检查ODS/原始层", "ods层是否有今天的数据入库？数据量是否正常？"),
    ("⑥ 检查Kafka/消息队列", "Kafka是否有消息积压？Consumer是否在消费？"),
    ("⑦ 检查数据采集", "Canal/Flume是否正常运行？业务库binlog是否在产生？"),
    ("⑧ 检查业务系统", "业务系统昨天是否正常运行？用户是否正常产生了数据？"),
]

for step, detail in checks:
    print(f"{step}")
    print(f"  → {detail}")
    print()

print("关键启示：延迟排查是从上往下还是从下往上？")
print("→ 从下往上（从业务系统到应用层），因为问题往往在源头。")
print("→ 但沟通时从上往下问（先确认是不是看板问题，避免误报）。")
print()

# ===== 步骤2.2：数据质量快速检查 =====

print("=" * 60)
print("数据质量快速检查SQL模板")
print("=" * 60)
print()

print("-- 检查1：今日数据量是否正常（对比昨日）")
print("SELECT ")
print("    dt,")
print("    COUNT(*) AS row_count,")
print("    COUNT(DISTINCT user_id) AS unique_users")
print("FROM dwd_order_detail")
print("WHERE dt >= DATE_SUB(CURRENT_DATE(), INTERVAL 7 DAY)")
print("GROUP BY dt")
print("ORDER BY dt DESC;")
print()

print("-- 检查2：是否有异常NULL值")
print("SELECT")
print("    dt,")
print("    SUM(CASE WHEN user_id IS NULL THEN 1 ELSE 0 END) AS null_user_id,")
print("    SUM(CASE WHEN amount IS NULL THEN 1 ELSE 0 END) AS null_amount,")
print("    COUNT(*) AS total")
print("FROM dwd_order_detail")
print("WHERE dt = DATE_SUB(CURRENT_DATE(), INTERVAL 1 DAY)")
print("GROUP BY dt;")
print()

print("-- 检查3：是否有重复数据")
print("SELECT")
print("    order_id,")
print("    COUNT(*) AS dup_count")
print("FROM dwd_order_detail")
print("WHERE dt = DATE_SUB(CURRENT_DATE(), INTERVAL 1 DAY)")
print("GROUP BY order_id")
print("HAVING COUNT(*) > 1")
print("LIMIT 10;")

### 步骤3：挑战练习——设计数据质量监控方案

In [ ]:
# ===== 步骤3：设计数据质量监控方案 =====

print("=" * 60)
print("挑战：为 dwd_order_detail 表设计数据质量监控方案")
print("=" * 60)
print()

print("监控维度1：完整性（数据有没有丢）")
print("  规则1：每日行数波动率 < 20%（对比过去7天均值）")
print("  规则2：每日唯一order_id数量 = 上游ods_order_log的唯一order_id数量")
print("  规则3：必要字段（order_id, user_id, amount, order_date）的NULL率 = 0%")
print()

print("监控维度2：准确性（数据对不对）")
print("  规则1：amount > 0（订单金额必须为正）")
print("  规则2：order_date <= CURRENT_DATE()（不能有未来日期）")
print("  规则3：user_id 存在于 dim_user 维表中（外键完整性）")
print()

print("监控维度3：一致性（口径对不对）")
print("  规则1：每日SUM(amount) 与业务库直接查询的偏差 < 1%")
print("  规则2：不同分区之间的数据不重复（order_id全局唯一）")
print()

print("监控维度4：时效性（数据有没有按时到）")
print("  规则1：dwd层数据在每天上午8:00前产出")
print("  规则2：ODS到DWD的延迟不超过2小时")
print()

print("实现建议：")
print("  1. 每周末运行一次完整的数据质量检查")
print("  2. 核心指标（行数、NULL率）每天检查")
print("  3. 发现异常自动发钉钉/企业微信告警")
print("  4. 把监控结果记录到数据质量看板，持续追踪")
print()

print("分析师的价值体现：")
print("  你不是在'等数据出错再修'——")
print("  你是在'数据出错之前就发现并告警'。")
print("  这就是从'取数工具人'到'数据建设者'的跃迁。")

## 📊 区块4：业务场景应用

### 场景1：运营的DAU和后台的DAU对不上

早上10点，运营总监在群里发了两张截图：
- 你给的数据看板：昨天DAU = 102万
- App后台实时面板：昨天在线用户峰值 = 98万

"差4万，哪个是对的？"

**你的排查过程**：

**第一步——确认口径**
- 你的DAU = 当天至少打开过一次App的去重用户数（按user_id去重），统计窗口是自然日0:00-23:59
- 后台的"在线用户" = 当前时刻保持连接的设备数（按device_id去重），统计窗口是瞬时值

这是两个**完全不同**的指标！你的DAU是"今天有多少人来过"，后台的是"现在有多少人在线"。就像一个商场——你统计的是"今天进了商场大门的人数"，后台统计的是"此刻在商场里的人数"。两者本就不该相等。

**第二步——确认去重维度**
- 你按user_id去重：一个用户在多台设备上登录只算一次
- 后台按device_id去重：一个用户用手机+平板登录算两次

如果平均每个用户有1.05个设备，那你按user_id算的102万 ≈ 后台按device_id算的107万... 等等，后台是98万？

**第三步——确认统计窗口**
- 你的DAU是"自然日0:00-23:59"
- 后台的"在线用户峰值"是"昨天任意时刻同时在线设备数的最大值"

关键差异找到了：DAU是"累计去重"——一个用户早上8点来、晚上9点也来，只算1次。在线峰值是"瞬时快照"——同一时刻最多有多少设备连着。这是两个完全不同的物理量，差4万太正常了。

**结论**：两个数字都对，只是定义不同。你在群里回复："两个指标口径不同。DAU是累计去重用户数（102万），后台是在线设备峰值（98万），前者是'来过的人'，后者是'同时在的人'。建议以后统一口径：对外说DAU，对内运维看在线峰值。"

**关键启示**："数据对不上"是分析师的日常。不要一上来就怀疑"数据错了"——80%的情况是"口径定义不同"。

### 场景2：昨天的数据到现在还没出来

下午2点，运营来催："今天的日报还没法写，昨天的数据还没出来！"

**你的排查过程**：

**第一步——确认是不是全部数据都没出**
```sql
-- 检查dwd层最新分区
SHOW PARTITIONS dwd_order_detail;
-- 发现：昨天的分区存在，但数据量为0
```

**第二步——往上游查**
```sql
-- 检查ods层
SHOW PARTITIONS ods_order_log;
-- 发现：昨天的分区也不存在！
```

**第三步——查调度系统**
- 登录调度平台（Airflow/DolphinScheduler）
- 发现：ods_order_log的同步任务昨晚失败了，因为Kafka消费者在凌晨2点挂掉，凌晨3点重试成功，但产生了3小时的数据积压

**第四步——估算恢复时间**
- Kafka积压量：约5000万条消息
- 消费速度：约100万条/分钟
- 预计清理积压需要50分钟
- 加上后续ETL任务（清洗+汇总），预计下午4点前数据产出

**第五步——通知运营**
"昨天数据延迟是因为上游Kafka消费者故障，目前正在追赶积压。预计今天下午4点前产出。如果急需昨天的数据，可以先看业务库的直接查询结果（但口径和数仓可能不完全一致）。"

**关键启示**：知道延迟的"在哪一层"和"为什么"，比知道"延迟了"本身重要10倍。前者让你能给出准确的时间预期，后者只会增加焦虑。

### 场景3：新来的分析师不知道表从哪来

团队新来了一个分析师，他看着数仓里300多张表，完全不知道每张表是什么意思、从哪来。

你作为老员工，给他画了一张"数据地图"：

```
数仓表命名规范速查

前缀含义：
  ods_ → 操作数据层（原始数据，照搬业务库）
  dwd_ → 明细数据层（清洗后，标准化）
  dws_ → 汇总数据层（轻度聚合，按天/人/品类）
  ads_ → 应用数据层（面向具体看板/报表）
  dim_ → 维度表（用户/商品/店铺的基础信息）
  tmp_ → 临时表（ETL中间结果，不要用）

后缀含义：
  _df → 全量快照表（每天一份完整数据）
  _di → 增量表（每天只存新增数据）
  _1d → 天粒度汇总
  _1w → 周粒度汇总
  _1m → 月粒度汇总

常用表速查：
  dwd_order_detail_df    → 订单明细（最常用）
  dwd_user_register_df   → 用户注册明细
  dws_user_daily_1d      → 用户日活汇总（日常分析首选）
  dws_category_daily_1d  → 品类日汇总（GMV分析首选）
  dim_user               → 用户维度表
  dim_product             → 商品维度表
```

**关键启示**：分析师不只是"用数据的人"，还应该是"让数据变得更好用的人"。如果你没有给新人写这样的文档——现在就是最好的时机。

### 场景4：业务库和数仓的数据差了3%

财务同事来找你："我从MySQL直接查的月GMV是8,500万，但你给的数仓数据是8,755万，差了255万（3%）。用哪个给投资人汇报？"

**排查过程**：

**第一步——确认时间范围**
- MySQL查询：`WHERE order_date BETWEEN '2026-01-01' AND '2026-01-31'`
- 数仓查询：`WHERE dt BETWEEN '2026-01-01' AND '2026-01-31'`
- 相同 ✓

**第二步——确认GMV口径**
- MySQL：`SUM(amount)` — 下单金额
- 数仓：`SUM(gmv_net)` — 实付金额+运费-退款
- 口径不同！

**第三步——对齐口径后重新对比**
- 按相同口径（下单金额）重新查询数仓：8,512万
- MySQL查询结果：8,500万
- 差异缩小到12万（0.14%）

**第四步——找到0.14%差异的原因**
- MySQL查询时，有3笔订单（合计12万）刚被物理删除（业务逻辑调整）
- 数仓通过binlog同步，这3笔订单在删除前已经被同步过去了
- 这是合理差异

**结论**："数仓和业务库差了3%是因为口径不同（下单金额 vs 实付GMV）。口径对齐后差异只有0.14%，是3笔在同步后被删除的订单导致的。用数仓数据给投资人汇报即可，我会在报告里标注GMV口径为'下单金额（含未支付）'。"

**关键启示**：面对"数据对不上"，永远先怀疑口径，再怀疑技术。口径对齐后，99%的"数据错误"会消失。

### 场景5：推动从"手动取数"到"数据产品化"

你发现团队每周都要手动跑同一个"用户留存分析"SQL，每次跑30分钟。你决定推动把这个分析"产品化"。

**你的推进步骤**：

1. **量化浪费**：每周5个分析师 × 30分钟 × 52周 = 130小时/年 = 约16个工作日
2. **提出方案**：在DWS层建一张`dws_user_retention_1d`表，每天自动更新
3. **评估成本**：ETL任务每天跑10分钟，占用集群资源约5%
4. **评估收益**：分析师从30分钟 × 5人/周 → 0秒（直接查表）
5. **推动落地**：在周会上提，拿到leader支持 → 和数仓同学对齐方案 → 提需求 → 跟进上线

一个月后，这张表上线了。全团队每周省下2.5小时的重复劳动。

> **关键启示**：这是数据分析师从"执行者"到"建设者"的关键跃迁——不只解决问题，还消除问题。

## ✅ 区块5：分级检验清单

### 🟢 初级检验（必须能独立完成的）
- [ ] 能画出数据从业务系统到分析看板的完整流转路径
- [ ] 能区分OLTP和OLAP，说出各自适用场景
- [ ] 能解释为什么不能直接在业务数据库上跑分析SQL
- [ ] 能说出ETL和ELT的区别

**答案提示**：
- 完整路径：业务系统 → 数据采集（Canal/Flume）→ 消息队列（Kafka）→ ETL处理 → 数仓存储（ODS→DWD→DWS→ADS）→ 数据应用
- OLTP面向实时业务操作（毫秒级），OLAP面向离线数据分析（秒到分钟级）
- 分析SQL占用大量IO/CPU会拖垮业务系统，OLTP和OLAP必须物理隔离
- ETL先转换再加载（传统），ELT先加载再转换（现代大数据）

### 🟡 中级检验（能解释给同事听的原理）
- [ ] 当两个数据源数字对不上时，能系统性地排查（口径→时间→去重→技术）
- [ ] 能解释Kafka在数据流转中的三个核心作用
- [ ] 能说出数据延迟可能发生在哪些环节
- [ ] 能设计简单的数据质量监控方案
- [ ] 能跟数仓工程师有效沟通数据需求

**答案提示**：
- 排查顺序：确认口径一致 → 确认时间窗口一致 → 确认去重逻辑一致 → 排查技术问题
- Kafka三大作用：削峰填谷、解耦生产者和消费者、支持多订阅
- 延迟环节：客户端上报、日志采集、Kafka积压、ETL任务失败、上游依赖未产出
- 监控四维度：完整性、准确性、一致性、时效性

### 🔴 高级检验（能举一反三的迁移能力）
- [ ] 能从分析师视角评估现有数据架构的不足并提出改进方案
- [ ] 能设计完整的"数据产品化"方案（从手动取数 → 自动产出）
- [ ] 能主动发现数据流转中的效率瓶颈并推动优化
- [ ] 能建立团队级的数据质量监控体系
- [ ] 能在数据延迟事件中担任"技术翻译"角色（业务方↔数仓工程师）

**答案提示**：
- 评估维度：时效性是否满足业务需求、关键指标是否有一致口径、常用分析是否已产品化
- 数据产品化步骤：量化浪费→提出方案→评估ROI→推动落地→持续优化
- 效率瓶颈：高频重复查询应进入DWS层、大表JOIN应预计算
- 质量体系：完整性/准确性/一致性/时效性四维监控 + 自动告警 + 质量看板
- 翻译角色：把业务的"数据怎么还没出来"翻译成数仓能处理的"ODS层昨天的分区有没有生成"

## 🔗 区块6：跨模块连接

| 连接方向 | 关联内容 | 为什么有关 |
|---------|---------|----------|
| **← 前置依赖** | 第1-3周的商业思维框架 | 理解数据流向是为了更好地应用那些框架——异动分析需要知道数据在哪，AARRR需要知道数据时效性 |
| **← 前置依赖** | 第1周SQL基础 | 你写的SQL是跑在数仓的哪一层？答案取决于你理解了哪层存了什么数据 |
| **→ 后续用途** | 明日数仓建模 | 今天知道数据要"分层"，明天学每层怎么建——分层逻辑是建模的基础 |
| **→ 后续用途** | 周四周五Hive与优化 | 理解了数据流转，才能理解Hive在大数据架构中的位置和优化策略 |
| **→ 后续用途** | 第5-6周SQL分析实战 | 写SQL时你会知道数据在哪一层、时效性如何、为什么慢 |
| **业务连接** | 所有数据质量排查 | 数据对不上的根因排查，必须从数据流转链路入手 |
| **业务连接** | 数据产品化 | 把高频分析需求固化为数仓表是分析师的核心贡献 |

### 横向连接：与其他分析领域的关联

**与数仓建模的关联**：
- 理解了数据流转的链路，才能合理设计分层
- 知道了数据延迟的来源，才能正确评估实时数仓的价值

**与数据质量的关联**：
- 数据质量问题往往发生在流转的某个环节
- 理解链路才能精准定位问题根源

**与数据治理的关联**：
- 数据血缘追踪是数据治理的核心
- 指标口径统一需要从数据源头做起

**与职业发展的关联**：
- "数据工程思维"是高级数据分析师的核心竞争力
- 理解数据基础设施能让你在团队中扮演更重要的角色

## 📝 区块7：当日小结

> **数据从业务系统到分析看板，经历了"业务库→采集→消息队列→ETL→数仓分层→应用"的漫长旅程。OLTP和OLAP必须分离，因为支撑业务和支持分析对资源的需求是冲突的。数据分析师的价值不在于"SQL写得多快"，而在于"理解数据背后的业务含义和流转过程"——这让你能在数据延迟时给出准确预判、在数据对不上时快速定位根因、在数据建设时提出有效方案。**

**三条金句带回家**：
1. 你不能在同一个系统上既跑业务又跑分析——就像你不能让同一辆车同时送货和赛车
2. 数据"对不上"时，80%是因为口径不同，先对齐口径再怀疑技术
3. 从"取数工具人"到"数据建设者"的关键跃迁——不只解决问题，而是消除问题

### 🔖 课后拓展

1. **练习**：找到你日常工作最常用的3张表，追溯它们的数据源头（从哪个业务系统来，经过了哪些ETL处理）

2. **思考**：如果你们公司要做实时数仓，哪些指标需要实时？哪些场景"T+1足够了"？为什么不是所有指标都实时？

3. **预习**：明天要学的数仓分层（ODS/DWD/DWS/ADS）是今天数据流转的进一步深化

4. **面试准备**：背诵以下面试题答案：
   - "你们公司的数据是怎么从业务库到数仓的？"
   - "OLTP和OLAP的区别是什么？"
   - "数据分析师为什么需要理解数据流转？"
   - "数据延迟了你怎么排查？"

---

*03analyst_thinking · Week 4 Day 1 · 数据从哪来*